# Week 6 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-06.md`](../../weeks/week-06.md) · **Given harness:** [`webharness.py`](webharness.py)

Teaching walkthrough of the week-6 studio (injection confirmed by a sound oracle,
then an LLM-vs-scanner duel). It **imports the reference answers from
[`solution.py`](solution.py)** — never re-pasting them — so what runs here is exactly
what `test_studio.py` grades. Verify: `python3 studios/_verify_solutions.py week-06`.

Laptop-only: `webharness` brings the `vuln-web` app up **in-process** on 127.0.0.1
(no Docker), and the LLM arm is scored from a **canned review string** (no Ollama).

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: week folder (for solution + engine) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-06"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import solution
# Importing webharness starts the in-process server on first get/post (no Docker).
from webharness import (regex_scanner, RAW_LLM_REVIEW, SOURCE, load_ground_truth)
from seclab.scan import compare_scanners, print_comparison
print("vuln-web harness ready (in-process, 127.0.0.1); ground truth + canned LLM review loaded")

## Task 1a — SQLi confirmed by an EXTRACTION oracle (not a vibe)

A login that behaves differently is only suggestive. **Proof** is extracting data you
shouldn't have — the admin canary `FLAG-sqli-...`. `contains_oracle` confirms only if
the canary comes *back* (an exfiltrated row), not if the payload is echoed. The benign
control (real user, wrong password) must land at 0/1 — a false positive in your tooling
poisons every number in your report. Mirrors `test_sqli_confirmed_by_canary_oracle` and
`test_benign_login_does_not_confirm`.

In [ ]:
print(inspect.getsource(solution.confirm_sqli))

findings = solution.confirm_sqli()
attacks = [f for f in findings if f.payload.family == "sqli"]
benign  = [f for f in findings if f.payload.family == "benign"]
assert attacks and all(f.confirmed for f in attacks), "every SQLi payload must leak the canary"
assert benign and not any(f.confirmed for f in benign), "benign login must NOT confirm"
print(f"SQLi: {len(attacks)}/{len(attacks)} payloads leaked the canary (extraction oracle)")
print(f"benign control confirmed: {any(f.confirmed for f in benign)}  (the oracle is not crying wolf)")

## Task 1b — the oracle-discipline crux: reflection is not execution

The same `<script>` marker fired at both endpoints. On the vulnerable `/` it survives
**unescaped** → the necessary condition for execution → **confirmed**. On the fixed
`/safe` (`html.escape`) it comes back as `&lt;script&gt;` → reflected but encoded →
**NOT confirmed**. A reflected string alone is never a finding. Mirrors
`test_reflected_but_escaped_is_not_a_confirmed_xss`.

In [ ]:
print(inspect.getsource(solution.confirm_xss))

vuln_f, safe_f = solution.confirm_xss()
assert vuln_f.confirmed, "reflected XSS on / must confirm (unescaped <script>)"
assert not safe_f.confirmed, "/safe html.escape()s the input — encoded reflection is NOT XSS"
print("vulnerable / :", "CONFIRMED" if vuln_f.confirmed else "no effect")
print("fixed /safe  :", "CONFIRMED" if safe_f.confirmed else "no effect", "(true negative)")
print("LESSON (axis 3): reflection is not execution.")

## Task 1c — command injection, confirmed on `injection_detected`

The `/ping` sink *simulates* execution (safe to run) but the bug — untrusted data
reaching a command context — is real. The sound oracle parses the JSON and confirms on
`injection_detected is True` (a shell metacharacter reached the command), not on the
echoed string. The benign host (no metacharacters) must not confirm. Mirrors
`test_cmdi_confirmed_and_benign_control_clean`.

In [ ]:
print(inspect.getsource(solution.confirm_cmdi))

findings = solution.confirm_cmdi()
attacks = [f for f in findings if f.payload.family == "cmdi"]
benign  = [f for f in findings if f.payload.family == "benign"]
assert attacks and all(f.confirmed for f in attacks), "metacharacter payloads must confirm"
assert benign and not any(f.confirmed for f in benign), "a plain host must NOT confirm"
print(f"cmdi: {len(attacks)}/{len(attacks)} confirmed; benign host confirmed: {any(f.confirmed for f in benign)}")

## Task 2 — the duel: parse the LLM arm into ScanResults, faithfully

`parse_llm_review` turns a model's free-text review into structured `ScanResult`s so it
is scored like the scanner. Parse **faithfully** — including the dubious findings; whether
they are real is decided by the ground truth, not the parser.

> The LLM arm here is a **canned review string** shipped in `webharness.py`. In the live
> studio you swap it for real `seclab.LLM` output — the parsing code does not change. If
> you use `LLM(backend="echo")`, remember the echo response is a **FAKE**, not a model;
> re-run against ollama/manual for a real result. Mirrors `test_llm_arm_parses_into_scanresults`.

In [ ]:
from seclab.llm import LLM
llm_client = LLM(backend="echo")   # headless plumbing check only
echo = llm_client.complete("find the vulns in vulnweb_app.py")
print("echo backend (FAKE, not a model):", echo.text[:60], "...\n")

print(inspect.getsource(solution.parse_llm_review))
llm = solution.parse_llm_review(RAW_LLM_REVIEW)
keys = {r.key() for r in llm}
for real in [("sqli", "do_login"), ("xss", "do_reflect"), ("cmdi", "do_ping")]:
    assert real in keys, f"LLM arm should have parsed {real}"
assert ("broken-access-control", "do_login") in keys, "must not drop the BAC claim"
print(f"parsed {len(llm)} LLM findings into ScanResults (faithfully, incl. the dubious ones):")
for r in llm:
    print("   ", r.key())

## The guarantee test — automation fails exactly where OWASP #1 lives

Both arms point at the **same source**, scored against the hand-verified ground truth.

- **(a)** The classical scanner's *completeness* FAILS: its line-oriented SQLi rule
  misses the 2-line query, so `recall < 1.0` and `("sqli","do_login")` is in its `missed`
  set. A genuine failure mode, not a contrivance.
- **(b)** NEITHER arm truthfully finds **Broken Access Control**: the scanner has no rule;
  the LLM's BAC "finding" is a FALSE POSITIVE (the app has no auth code). Access control
  needs app *intent* a pattern/LLM guess can't supply — which is why #1 is #1.

Stages `test_scanner_misses_sqli_and_neither_finds_bac`.

In [ ]:
gt = load_ground_truth()
cmp = compare_scanners(
    {"regex-scanner": regex_scanner(SOURCE), "llm": solution.parse_llm_review(RAW_LLM_REVIEW)},
    gt)
print_comparison(cmp)

scanner, llm_res = cmp["regex-scanner"], cmp["llm"]
bac = ("broken-access-control", "do_login")
assert scanner["recall"] < 1.0 and ("sqli", "do_login") in scanner["missed"]
assert bac not in gt and bac in llm_res["false_positives"]
scanner_keys = {r.key() for r in regex_scanner(SOURCE)}
assert bac not in scanner_keys
print("\nGUARANTEE-FAIL: scanner MISSED the SQLi (2-line query vs 1-line rule);")
print("NEITHER arm found Broken Access Control (LLM hallucinated it, scanner has no rule).")
print("LESSON: automation is worst at OWASP-2021 #1 — it is about intent, not syntax.")

## Control Scorecard + disclosure (Task 3)

| | Threat (axis 1) | Guarantee & condition (axis 2) | Coverage (axis 4/completeness) |
|---|---|---|---|
| Oracle-confirmed SQLi | attacker at `/login` | *finding is real* **iff** a sound oracle sees the canary leak | high — extraction is unambiguous |
| Regex scanner | untrusted code review | catches *single-line* patterns | **misses** multi-line sinks (recall < 1) |
| LLM reviewer | untrusted code review | high recall | **hallucinates** (BAC false positive) — precision cost |

**Responsible-disclosure note (2 lines, for an app you were AUTHORIZED to test):**
> SQL injection at `POST /login` (`user` field, string-formatted query in `do_login`) —
> an auth-bypass leaks the admin secret row. Fix: parameterize the query
> (`WHERE user=? AND pw=?`). See `resources/ethics-and-scope.md` for what authorization means.